In [ ]:
from pathlib import Path
import zipfile

import pandas as pd
import matplotlib.pyplot as plt

FILE_PATH = Path("service_level_metrics.csv")  # Or Path("results.zip")

if FILE_PATH.suffix.lower() == ".zip":
    with zipfile.ZipFile(FILE_PATH) as archive:
        matches = [
            name for name in archive.namelist()
            if name.endswith("results/raw/service_level_metrics.csv")
        ]
        if not matches:
            raise FileNotFoundError(
                "The ZIP does not contain results/raw/service_level_metrics.csv."
            )
        if len(matches) > 1:
            raise ValueError(f"Found multiple raw service metric CSVs in the ZIP: {matches}")
        with archive.open(matches[0]) as file:
            metrics = pd.read_csv(file)
else:
    metrics = pd.read_csv(FILE_PATH)

# Keep only the HBNN single-training, bivariate run.
metrics = metrics[metrics["model"] == "S-B-HBNN"].copy()
if metrics.empty:
    raise ValueError("No S-B-HBNN rows found. Check that you downloaded the raw service metrics CSV.")

# Average service metrics across traces, if the file contains more than one.
mean_metrics = (
    metrics.groupby(["resource", "service_level"], as_index=False)
    .agg(SR=("SR", "mean"), TPR=("TPR", "mean"))
)

# Target confidence/service level vs. actual success rate.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, resource in zip(axes, ["avgcpu", "avgmem"]):
    data = mean_metrics[mean_metrics["resource"] == resource].sort_values("service_level")
    ax.plot(data["service_level"] * 100, data["SR"], marker="o", label="HBNN")
    ax.plot([90, 99.5], [90, 99.5], "k--", label="Ideal calibration")
    ax.set(title=f"{resource}: confidence vs. SR",
           xlabel="Target confidence / service level (%)")
    ax.grid(alpha=0.25)
    ax.legend()
axes[0].set_ylabel("Success rate, SR (%)")
fig.tight_layout()
fig.savefig("hbnn_confidence_vs_sr.png", dpi=150)
plt.show()

# Total predicted resources vs. success rate.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, resource in zip(axes, ["avgcpu", "avgmem"]):
    data = mean_metrics[mean_metrics["resource"] == resource].sort_values("TPR")
    ax.plot(data["TPR"], data["SR"], marker="o", label="HBNN")
    ax.set(title=f"{resource}: TPR vs. SR",
           xlabel="Predicted resources / actual demand, TPR (%)")
    ax.grid(alpha=0.25)
    ax.legend()
axes[0].set_ylabel("Success rate, SR (%)")
fig.tight_layout()
fig.savefig("hbnn_tpr_vs_sr.png", dpi=150)
plt.show()